In [ ]:
# Se deben importar las librerías usadas durante el análisis.

from pathlib import Path
import pandas as pd
import plotly.express as px

In [ ]:
# Se deben definir y verificar las rutas de los dos agregados reales.

DATA_DIR = Path('../data')
DAY_HOUR_PATH = DATA_DIR / 'flights_by_carrier_day_hour.csv.gz'
MONTHLY_PATH = DATA_DIR / 'flights_by_carrier_month.csv.gz'
assert DAY_HOUR_PATH.exists()
assert MONTHLY_PATH.exists()

In [ ]:
# Se debe cargar y verificar el agregado por aerolínea, día y hora programada.

day_hour = pd.read_csv(DAY_HOUR_PATH)
day_hour.info()
day_hour.head()

In [ ]:
# Se debe cargar y verificar el agregado mensual por aerolínea.

monthly = pd.read_csv(MONTHLY_PATH)
monthly.info()
monthly.head()

In [ ]:
# Se deben explorar las medidas numéricas del agregado por día y hora.

day_hour.describe().T

In [ ]:
# Se deben explorar las medidas numéricas del agregado mensual.

monthly.describe().T

In [ ]:
# Se debe verificar que ninguno de los dos agregados tenga valores faltantes.

assert day_hour.isna().sum().sum() == 0
assert monthly.isna().sum().sum() == 0

In [ ]:
# Se deben definir las medidas aditivas y las tasas con sus denominadores correctos.

metric_columns = ['scheduled_flights', 'cancelled_flights', 'operated_flights', 'delayed_departure_15_flights', 'positive_departure_delay_minutes']
def add_rates(frame):
    result = frame.copy()
    result['cancellation_rate'] = result['cancelled_flights'] / result['scheduled_flights']
    result['delay_rate'] = result['delayed_departure_15_flights'] / result['operated_flights']
    result['mean_positive_delay_minutes'] = result['positive_departure_delay_minutes'] / result['operated_flights']
    return result

In [ ]:
# Se debe verificar que reagrupar día y hora produce exactamente el agregado mensual.

recomputed = day_hour.groupby(['year', 'month', 'reporting_airline'])[metric_columns].sum().reset_index()
recomputed = recomputed.sort_values(['year', 'month', 'reporting_airline']).reset_index(drop=True)
provided = monthly.sort_values(['year', 'month', 'reporting_airline']).reset_index(drop=True)
pd.testing.assert_frame_equal(recomputed, provided)

In [ ]:
# ¿Cuál es el KPI nacional de demora, cancelación y minutos positivos?

overall = add_rates(monthly[metric_columns].sum().to_frame().T)
overall[['scheduled_flights', 'cancelled_flights', 'operated_flights', 'delay_rate', 'cancellation_rate', 'mean_positive_delay_minutes']].round(3)

In [ ]:
# ¿Cómo evoluciona mensualmente la tasa nacional de demora?

monthly_overall = add_rates(monthly.groupby(['year', 'month'])[metric_columns].sum().reset_index())
monthly_overall['period'] = pd.to_datetime(dict(year=monthly_overall.year, month=monthly_overall.month, day=1))
fig = px.line(monthly_overall, x='period', y='delay_rate', markers=True, hover_data=['operated_flights', 'cancellation_rate'], title='Evolución mensual de la tasa nacional de demora')
fig.update_yaxes(tickformat='.0%')
fig.update_layout(template='plotly_white', showlegend=False)
fig.show()

In [ ]:
# ¿Qué aerolíneas combinan mayor tasa de demora, volumen y cancelación?

carrier_summary = add_rates(monthly.groupby('reporting_airline')[metric_columns].sum().reset_index())
carrier_summary = carrier_summary.sort_values('delay_rate', ascending=False)
top_carriers_by_volume = carrier_summary.nlargest(5, 'operated_flights')['reporting_airline'].tolist()
carrier_summary[['reporting_airline', 'operated_flights', 'delay_rate', 'cancellation_rate']].round(3)

In [ ]:
# ¿En qué aerolíneas se concentra la mayor tasa de demora?

fig = px.bar(carrier_summary, x='reporting_airline', y='delay_rate', hover_data=['operated_flights', 'delayed_departure_15_flights'], title='Tasa de demora por aerolínea')
fig.update_yaxes(tickformat='.0%')
fig.update_layout(template='plotly_white', showlegend=False)
fig.show()

In [ ]:
# ¿Qué días y horas programadas concentran las mayores tasas de demora?

day_names = {1:'Lunes', 2:'Martes', 3:'Miércoles', 4:'Jueves', 5:'Viernes', 6:'Sábado', 7:'Domingo'}
calendar_hour = add_rates(day_hour.groupby(['day_of_week', 'scheduled_departure_hour'])[metric_columns].sum().reset_index())
rate_matrix = calendar_hour.pivot(index='day_of_week', columns='scheduled_departure_hour', values='delay_rate')
fig = px.imshow(rate_matrix, color_continuous_scale='Oranges', aspect='auto', title='Tasa de demora por día y hora')
fig.update_yaxes(tickmode='array', tickvals=list(day_names), ticktext=list(day_names.values()))
fig.update_layout(template='plotly_white', coloraxis_colorbar_tickformat='.0%')
fig.show()

In [ ]:
# ¿Qué segmentos aerolínea–día–hora se deben priorizar para investigar y reducir las demoras, sin ignorar su volumen?

minimum_operated_flights = 25_000
critical = add_rates(day_hour.groupby(['reporting_airline', 'day_of_week', 'scheduled_departure_hour'])[metric_columns].sum().reset_index())
critical = critical[critical.operated_flights >= minimum_operated_flights]
critical['segment'] = critical.reporting_airline + ' — ' + critical.day_of_week.map(day_names) + ' — ' + critical.scheduled_departure_hour.map(lambda hour: f'{hour:02d}:00')
critical = critical.nlargest(10, 'delay_rate')
fig = px.bar(critical.sort_values('delay_rate'), x='delay_rate', y='segment', orientation='h', hover_data=['operated_flights', 'cancellation_rate'], title='Diez segmentos prioritarios')
fig.update_xaxes(tickformat='.0%')
fig.update_layout(template='plotly_white', showlegend=False)
fig.show()

In [ ]:
# ¿La evolución de las aerolíneas de mayor volumen revela un patrón estacional?

seasonality = add_rates(monthly.groupby(['month', 'reporting_airline'])[metric_columns].sum().reset_index())
seasonality = seasonality[seasonality.reporting_airline.isin(top_carriers_by_volume)]
fig = px.line(seasonality, x='month', y='delay_rate', color='reporting_airline', markers=True, hover_data=['operated_flights'], title='Patrón estacional de las aerolíneas con mayor volumen')
fig.update_xaxes(dtick=1)
fig.update_yaxes(tickformat='.0%')
fig.update_layout(template='plotly_white')
fig.show()